# Evaluación — Red Neuronal Multicapa (MLP) para clasificación
## Dataset: *Bank Marketing* (UCI Machine Learning Repository)

### Contexto del problema

Un banco portugués realizó **campañas de marketing telefónico** para ofrecer a sus clientes un **depósito a plazo**. Por cada cliente contactado se registraron sus datos personales, su situación financiera y el historial de contactos de la campaña.

**Objetivo:** construir un modelo que prediga si un cliente **contratará el depósito a plazo** (`y = "yes"`) o **no** (`y = "no"`).

| | |
|---|---|
| **Tipo de problema** | Clasificación **binaria** |
| **Registros** | 45.211 clientes |
| **Variables de entrada** | 16 (7 numéricas y 9 categóricas) |
| **Variable objetivo** | `y` ("yes" / "no") |
| **Fuente** | [UCI Machine Learning Repository — Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing) |
| **Licencia** | CC BY 4.0 |
| **Cita** | Moro, S., Cortez, P. & Rita, P. (2014). *A data-driven approach to predict the success of bank telemarketing.* Decision Support Systems, 62, 22–31. |

## ¿Qué deben demostrar?

1. Cargar y **preprocesar** correctamente los datos.
2. **Diseñar e implementar** una arquitectura MLP (TensorFlow/Keras o PyTorch).
3. Configurar **parámetros de entrenamiento**: épocas, tasa de aprendizaje y tamaño de batch.
4. **Comparar** distintas funciones de **activación** y funciones de **pérdida**.
5. Aplicar técnicas de **optimización** y **regularización**.
6. **Evaluar** el desempeño con **Accuracy, Precision, Recall y F1-Score**.
7. **Analizar** los resultados y **justificar** las mejoras realizadas al modelo.

> Este notebook **solo carga y describe los datos**. Todo el preprocesamiento, el modelo y el análisis son parte de su trabajo.

## 1. Cargar los datos

Ejecuta **una** de las dos opciones.

**Opción A (recomendada): descarga directa desde UCI.** Funciona en Google Colab y en Jupyter local. Solo requiere internet.

In [ ]:
import io
import zipfile
import urllib.request

import pandas as pd
import matplotlib.pyplot as plt

URL_UCI = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

with urllib.request.urlopen(URL_UCI) as resp:
    zip_externo = zipfile.ZipFile(io.BytesIO(resp.read()))
zip_interno = zipfile.ZipFile(io.BytesIO(zip_externo.read("bank.zip")))   # el archivo viene comprimido dos veces
df = pd.read_csv(zip_interno.open("bank-full.csv"), sep=";")

print("Datos cargados:", df.shape[0], "filas x", df.shape[1], "columnas")

**Opción B: desde el archivo `bank_marketing.csv` entregado por el docente.**

- **En Colab:** sube el archivo con el ícono de carpeta (panel izquierdo → *Subir*) y ejecuta la celda.
- **En Jupyter local:** deja el CSV en la misma carpeta que este notebook.
- **Desde un enlace** (Google Drive / GitHub): pega el enlace en `ENLACE`. Si es de Google Drive, el archivo debe estar compartido como *"Cualquier persona con el enlace"*.

In [ ]:
import os

ENLACE = ""   # opcional: pega aquí el enlace entregado por el docente

def enlace_descarga(url):
    # Convierte un enlace compartido de Google Drive en un enlace de descarga directa
    if "drive.google.com" in url and "/d/" in url:
        file_id = url.split("/d/")[1].split("/")[0]
        return f"https://drive.google.com/uc?export=download&id={file_id}"
    return url

# Esta opción solo reemplaza df si existe un enlace o un archivo local.
# Así "Ejecutar todo" no falla si se usó la opción A.
if ENLACE:
    df = pd.read_csv(enlace_descarga(ENLACE))
    print("Datos cargados desde enlace:", df.shape[0], "filas x", df.shape[1], "columnas")
elif os.path.exists("bank_marketing.csv"):
    df = pd.read_csv("bank_marketing.csv")
    print("Datos cargados desde CSV:", df.shape[0], "filas x", df.shape[1], "columnas")
else:
    print("No se encontró CSV local. Se mantiene el dataset cargado con la opción A.")


## 2. Diccionario de variables

**Datos del cliente**

| Variable | Tipo | Descripción |
|---|---|---|
| `age` | numérica | Edad |
| `job` | categórica | Tipo de trabajo (`admin.`, `blue-collar`, `entrepreneur`, `housemaid`, `management`, `retired`, `self-employed`, `services`, `student`, `technician`, `unemployed`, `unknown`) |
| `marital` | categórica | Estado civil (`married`, `single`, `divorced` — "divorced" incluye viudos) |
| `education` | categórica | Nivel educacional (`primary`, `secondary`, `tertiary`, `unknown`) |
| `default` | binaria | ¿Tiene créditos en mora? (`yes` / `no`) |
| `balance` | numérica | Saldo promedio anual, en euros (puede ser negativo) |
| `housing` | binaria | ¿Tiene crédito hipotecario? |
| `loan` | binaria | ¿Tiene crédito de consumo? |

**Último contacto de la campaña actual**

| Variable | Tipo | Descripción |
|---|---|---|
| `contact` | categórica | Medio de contacto (`cellular`, `telephone`, `unknown`) |
| `day` | numérica | Día del mes del último contacto |
| `month` | categórica | Mes del último contacto (`jan` … `dec`) |
| `duration` | numérica | Duración del último contacto, en segundos ⚠️ *(ver nota importante más abajo)* |

**Otros atributos**

| Variable | Tipo | Descripción |
|---|---|---|
| `campaign` | numérica | N° de contactos realizados a este cliente en esta campaña (incluye el último) |
| `pdays` | numérica | Días transcurridos desde el último contacto de una campaña **anterior** (`-1` = nunca fue contactado antes) |
| `previous` | numérica | N° de contactos realizados antes de esta campaña |
| `poutcome` | categórica | Resultado de la campaña anterior (`success`, `failure`, `other`, `unknown`) |

**Variable objetivo**

| Variable | Tipo | Descripción |
|---|---|---|
| `y` | binaria | ¿El cliente contrató el depósito a plazo? (`yes` / `no`) |

## 3. Primera mirada a los datos

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

### Distribución de la variable objetivo

In [ ]:
conteo = df["y"].value_counts()
porcentaje = df["y"].value_counts(normalize=True).mul(100).round(1)
display(pd.DataFrame({"cantidad": conteo, "porcentaje (%)": porcentaje}))

conteo.plot(kind="bar", color=["tab:gray", "tab:green"], figsize=(5, 3.5), rot=0,
            title="¿Contrató el depósito a plazo?")
plt.ylabel("clientes")
plt.show()

### Valores "unknown"

El dataset **no tiene celdas vacías**, pero varias columnas categóricas usan el texto `"unknown"` cuando no se conoce el dato.

In [ ]:
print("Celdas vacías (NaN):", df.isna().sum().sum())
categoricas = [c for c in df.columns if df[c].dtype == "object" or str(df[c].dtype) == "str"]
unknown = {c: int((df[c] == "unknown").sum()) for c in categoricas}
pd.Series(unknown, name="cantidad de 'unknown'").to_frame().query("`cantidad de 'unknown'` > 0")

## 4. Consideraciones importantes

1. **Clases desbalanceadas.** Solo ~12 % de los clientes contrató el depósito. Un modelo que responda siempre `"no"` obtiene **~88 % de Accuracy** sin haber aprendido nada. Por eso **Precision, Recall y F1-Score** de la clase `"yes"` son fundamentales para evaluar los modelos.
2. **La variable `duration`.** La duración de la llamada solo se conoce **después** de hacerla, y cuando `duration = 0` el resultado siempre es `"no"`. Los autores del dataset recomiendan **no usarla** si el objetivo es un modelo predictivo realista (uno que se use *antes* de llamar al cliente). Decidan si la incluyen o no y **justifiquen** su decisión en el análisis.
3. **Variables categóricas.** La red neuronal necesita números: hay que transformar las variables de texto.
4. **Escalas muy distintas.** Por ejemplo, `balance` va de miles de euros negativos a más de 100.000, mientras que `previous` suele estar entre 0 y 5.
5. **`pdays = -1`** no es un número de días: significa "nunca contactado antes".
6. **Separación de datos.** Todo lo que el preprocesamiento "aprende" (medias, escalas, categorías) debe calcularse **solo con los datos de entrenamiento**.

## 5. Su trabajo comienza aquí ⬇️

Sugerencia de estructura para el notebook de entrega:

1. Análisis exploratorio y preprocesamiento
2. Separación en entrenamiento / validación / prueba
3. Modelo base (MLP) y configuración de entrenamiento
4. Comparación de funciones de activación y de pérdida
5. Optimización y regularización
6. Evaluación final: Accuracy, Precision, Recall, F1-Score y matriz de confusión
7. Análisis de resultados y justificación de las mejoras

## 6. Preparación y preprocesamiento

La variable objetivo es `y`, donde `no = 0` y `yes = 1`.

Se decidió **excluir `duration`** porque la duración de la llamada solo se conoce después de realizarla. El objetivo del modelo es apoyar una decisión antes del contacto, por lo que incluir esta variable produciría fuga de información y un rendimiento poco realista.

Los valores `"unknown"` se mantienen como una categoría válida, porque significan que la información no estaba disponible. No se eliminan esos registros para no perder una cantidad importante de observaciones.

También se trata de forma especial `pdays = -1`. En el dataset este valor no representa “menos un día”, sino que indica que el cliente nunca había sido contactado antes. Para conservar esa información se crea una variable binaria llamada `contactado_antes` y luego `pdays = -1` se reemplaza por 0 antes del escalamiento.


In [ ]:
# Copia del dataset para no modificar el original
datos = df.copy()

# Variable objetivo: no = 0, yes = 1
datos["y"] = datos["y"].map({"no": 0, "yes": 1})

# Se elimina duration para evitar fuga de información
X = datos.drop(columns=["y", "duration"]).copy()
y = datos["y"].copy()

# Tratamiento de pdays:
# 1 = tuvo contacto en una campaña anterior, 0 = nunca fue contactado
X["contactado_antes"] = (X["pdays"] != -1).astype(int)
X["pdays"] = X["pdays"].replace(-1, 0)

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)
print("\nDistribución de la variable objetivo:")
display(y.value_counts().rename(index={0:"no",1:"yes"}).to_frame("cantidad"))
display((y.value_counts(normalize=True)*100).round(2).rename(index={0:"no",1:"yes"}).to_frame("porcentaje"))


### 6.1 Separación en entrenamiento, validación y prueba

Se utiliza una división de **60% entrenamiento, 20% validación y 20% prueba**.

- **Entrenamiento:** se usa para que la red aprenda sus pesos.
- **Validación:** se usa para comparar configuraciones e hiperparámetros.
- **Prueba:** se reserva hasta el final para medir la capacidad de generalización.

Se utiliza `stratify` para conservar aproximadamente la misma proporción de las clases `yes` y `no` en los tres subconjuntos. Esto es importante porque la clase positiva representa cerca del 12% de los datos.


In [ ]:
from sklearn.model_selection import train_test_split

# 80% temporal y 20% prueba
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Del 80% temporal: 75% train y 25% validación
# Resultado final: 60% train, 20% validación, 20% test
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print("Entrenamiento:", X_train.shape)
print("Validación:", X_val.shape)
print("Prueba:", X_test.shape)

print("\nPorcentaje de clase positiva:")
print("Train:", round(y_train.mean()*100, 2), "%")
print("Val:  ", round(y_val.mean()*100, 2), "%")
print("Test: ", round(y_test.mean()*100, 2), "%")


### 6.2 Transformación de variables

Las variables numéricas se estandarizan con `StandardScaler`, ya que presentan escalas muy distintas. Esto ayuda a que la optimización por descenso de gradiente sea más estable.

Las variables categóricas se transforman mediante `OneHotEncoder`. Se utiliza `handle_unknown="ignore"` para que una categoría que no aparezca durante el entrenamiento no genere un error al transformar validación o prueba.

El preprocesador se **ajusta únicamente con el conjunto de entrenamiento**. Los conjuntos de validación y prueba solo se transforman. De esta forma se evita fuga de información.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

columnas_numericas = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
columnas_categoricas = X_train.select_dtypes(include=["object", "string"]).columns.tolist()

print("Variables numéricas:", columnas_numericas)
print("Variables categóricas:", columnas_categoricas)

preprocesador = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), columnas_numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), columnas_categoricas)
    ]
)

# fit solo en entrenamiento
X_train_proc = preprocesador.fit_transform(X_train)
X_val_proc = preprocesador.transform(X_val)
X_test_proc = preprocesador.transform(X_test)

print("\nDimensiones después del preprocesamiento:")
print("Train:", X_train_proc.shape)
print("Val:  ", X_val_proc.shape)
print("Test: ", X_test_proc.shape)

print("\nNaN después del preprocesamiento:",
      int(__import__("numpy").isnan(X_train_proc).sum()
          + __import__("numpy").isnan(X_val_proc).sum()
          + __import__("numpy").isnan(X_test_proc).sum()))


### 6.3 Tratamiento del desbalance

El conjunto está desbalanceado: la mayoría de los clientes pertenece a la clase `no`. Un clasificador que predijera siempre `no` podría alcanzar un Accuracy alto sin detectar clientes que realmente contratarían el depósito.

Por este motivo se calculan **pesos de clase** usando solamente `y_train`. La clase minoritaria recibe un peso mayor durante el entrenamiento. Los mismos pesos se mantienen en todos los experimentos para que las comparaciones sean consistentes.

Para seleccionar configuraciones se dará mayor importancia a **F1-Score**, sin dejar de revisar Precision, Recall y Accuracy.


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

clases = np.unique(y_train)
pesos = compute_class_weight(
    class_weight="balanced",
    classes=clases,
    y=y_train
)

pesos_clase = {int(c): float(p) for c, p in zip(clases, pesos)}
print("Pesos de clase:", pesos_clase)


## 7. Arquitectura MLP base

La red base tiene dos capas ocultas:

- **64 neuronas + ReLU**
- **32 neuronas + ReLU**
- **1 neurona de salida + Sigmoid**

Se eligieron dos capas ocultas porque el problema es tabular y no requiere una arquitectura excesivamente profunda. La reducción de 64 a 32 neuronas permite aprender combinaciones de las variables de entrada de forma progresiva, manteniendo una capacidad suficiente sin aumentar innecesariamente la complejidad.

`ReLU` se utiliza inicialmente en las capas ocultas porque introduce no linealidad y suele favorecer una convergencia eficiente. La salida usa `sigmoid` porque se trata de una clasificación binaria y necesitamos una probabilidad entre 0 y 1.

La pérdida inicial es `binary_crossentropy`, apropiada para clasificación binaria. El optimizador base es Adam.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

tf.random.set_seed(42)
np.random.seed(42)

def obtener_optimizador(nombre="adam", learning_rate=0.001):
    nombre = nombre.lower()
    if nombre == "adam":
        return keras.optimizers.Adam(learning_rate=learning_rate)
    if nombre == "rmsprop":
        return keras.optimizers.RMSprop(learning_rate=learning_rate)
    if nombre == "sgd":
        return keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    raise ValueError("Optimizador no reconocido")

def crear_modelo(
    activacion="relu",
    learning_rate=0.001,
    perdida="binary_crossentropy",
    optimizador="adam",
    regularizacion=False
):
    modelo = keras.Sequential(name="MLP_BankMarketing")
    modelo.add(layers.Input(shape=(X_train_proc.shape[1],)))

    if regularizacion:
        modelo.add(layers.Dense(
            64,
            activation=activacion,
            kernel_regularizer=regularizers.l2(0.001)
        ))
        modelo.add(layers.BatchNormalization())
        modelo.add(layers.Dropout(0.30))

        modelo.add(layers.Dense(
            32,
            activation=activacion,
            kernel_regularizer=regularizers.l2(0.001)
        ))
        modelo.add(layers.Dropout(0.20))
    else:
        modelo.add(layers.Dense(64, activation=activacion))
        modelo.add(layers.Dense(32, activation=activacion))

    # Salida binaria
    modelo.add(layers.Dense(1, activation="sigmoid"))

    modelo.compile(
        optimizer=obtener_optimizador(optimizador, learning_rate),
        loss=perdida,
        metrics=["accuracy"]
    )
    return modelo

modelo_base = crear_modelo()
modelo_base.summary()


### 7.1 Configuración inicial de entrenamiento

Se parte con una configuración base de:

- **30 épocas**
- **batch size = 32**
- **learning rate = 0.001**
- **Adam**
- **ReLU**
- **Binary Crossentropy**

Estos valores funcionan como punto de referencia. Luego se modifica un parámetro a la vez y se compara su efecto usando el conjunto de validación.


In [ ]:
EPOCHS_BASE = 30
BATCH_BASE = 32
LR_BASE = 0.001

hist_base = modelo_base.fit(
    X_train_proc, y_train,
    validation_data=(X_val_proc, y_val),
    epochs=EPOCHS_BASE,
    batch_size=BATCH_BASE,
    class_weight=pesos_clase,
    verbose=0
)

print("Entrenamiento base terminado")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
plt.plot(hist_base.history["accuracy"], label="Entrenamiento")
plt.plot(hist_base.history["val_accuracy"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Modelo base - Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(8,4))
plt.plot(hist_base.history["loss"], label="Entrenamiento")
plt.plot(hist_base.history["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Modelo base - Pérdida")
plt.legend()
plt.show()


## 8. Experimentos controlados de hiperparámetros

Para cumplir una comparación controlada, en cada bloque se modifica **un parámetro a la vez**. El conjunto de prueba no se utiliza durante esta etapa.

Las configuraciones se comparan con Accuracy, Precision, Recall y F1-Score de la clase positiva (`yes`). Se selecciona principalmente por F1 porque entrega un equilibrio entre Precision y Recall en un problema desbalanceado.


In [ ]:
def calcular_metricas(y_real, probabilidades, umbral=0.5):
    pred = (probabilidades >= umbral).astype(int)
    return {
        "accuracy": accuracy_score(y_real, pred),
        "precision": precision_score(y_real, pred, zero_division=0),
        "recall": recall_score(y_real, pred, zero_division=0),
        "f1": f1_score(y_real, pred, zero_division=0)
    }

def ejecutar_prueba(
    nombre,
    epochs=30,
    batch_size=32,
    learning_rate=0.001,
    activacion="relu",
    perdida="binary_crossentropy",
    optimizador="adam",
    regularizacion=False
):
    tf.keras.backend.clear_session()
    tf.random.set_seed(42)
    np.random.seed(42)

    modelo = crear_modelo(
        activacion=activacion,
        learning_rate=learning_rate,
        perdida=perdida,
        optimizador=optimizador,
        regularizacion=regularizacion
    )

    historia = modelo.fit(
        X_train_proc, y_train,
        validation_data=(X_val_proc, y_val),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=pesos_clase,
        verbose=0
    )

    prob_val = modelo.predict(X_val_proc, verbose=0).ravel()
    m = calcular_metricas(y_val, prob_val)

    resultado = {
        "prueba": nombre,
        "epochs": epochs,
        "batch": batch_size,
        "lr": learning_rate,
        "activacion": activacion,
        "perdida": perdida,
        "optimizador": optimizador,
        "regularizacion": regularizacion,
        **m
    }
    return resultado, historia

def graficar_historias(historias, metrica, titulo):
    plt.figure(figsize=(8,4))
    for etiqueta, historia in historias.items():
        plt.plot(historia.history[metrica], label=str(etiqueta))
    plt.xlabel("Época")
    plt.ylabel(metrica)
    plt.title(titulo)
    plt.legend()
    plt.show()


### 8.1 Experimento: número de épocas

Se prueban 15, 30 y 50 épocas manteniendo iguales el batch, learning rate, activación, pérdida y optimizador.


In [ ]:
import pandas as pd
from IPython.display import display, Markdown

resultados_epochs = []
historias_epochs = {}

for ep in [15, 30, 50]:
    r, h = ejecutar_prueba(
        nombre=f"epochs_{ep}",
        epochs=ep,
        batch_size=32,
        learning_rate=0.001,
        activacion="relu",
        perdida="binary_crossentropy",
        optimizador="adam"
    )
    resultados_epochs.append(r)
    historias_epochs[ep] = h

tabla_epochs = pd.DataFrame(resultados_epochs)
display(tabla_epochs[["prueba","accuracy","precision","recall","f1"]].round(4))

graficar_historias(
    historias_epochs, "val_loss",
    "Impacto del número de épocas - Pérdida de validación"
)

mejor_epochs = int(tabla_epochs.loc[tabla_epochs["f1"].idxmax(), "epochs"])
mejor_f1_epochs = tabla_epochs["f1"].max()

display(Markdown(
    f"**Análisis:** la configuración con **{mejor_epochs} épocas** obtuvo el mayor "
    f"F1-Score de validación en esta prueba (**{mejor_f1_epochs:.4f}**). "
    "Se utiliza este valor en el siguiente experimento."
))


### 8.2 Experimento: tamaño de batch

Se prueban batch size 16, 32 y 64. Se mantiene la mejor cantidad de épocas obtenida en el experimento anterior y el resto de los parámetros permanece fijo.


In [ ]:
resultados_batch = []
historias_batch = {}

for b in [16, 32, 64]:
    r, h = ejecutar_prueba(
        nombre=f"batch_{b}",
        epochs=mejor_epochs,
        batch_size=b,
        learning_rate=0.001,
        activacion="relu",
        perdida="binary_crossentropy",
        optimizador="adam"
    )
    resultados_batch.append(r)
    historias_batch[b] = h

tabla_batch = pd.DataFrame(resultados_batch)
display(tabla_batch[["prueba","accuracy","precision","recall","f1"]].round(4))

graficar_historias(
    historias_batch, "val_loss",
    "Impacto del tamaño de batch - Pérdida de validación"
)

mejor_batch = int(tabla_batch.loc[tabla_batch["f1"].idxmax(), "batch"])
mejor_f1_batch = tabla_batch["f1"].max()

display(Markdown(
    f"**Análisis:** el batch **{mejor_batch}** presentó el mayor F1-Score de validación "
    f"(**{mejor_f1_batch:.4f}**). Se mantiene en las pruebas siguientes."
))


### 8.3 Experimento: tasa de aprendizaje

Se comparan `0.01`, `0.001` y `0.0001`, manteniendo los demás parámetros seleccionados hasta este punto.


In [ ]:
resultados_lr = []
historias_lr = {}

for lr in [0.01, 0.001, 0.0001]:
    r, h = ejecutar_prueba(
        nombre=f"lr_{lr}",
        epochs=mejor_epochs,
        batch_size=mejor_batch,
        learning_rate=lr,
        activacion="relu",
        perdida="binary_crossentropy",
        optimizador="adam"
    )
    resultados_lr.append(r)
    historias_lr[lr] = h

tabla_lr = pd.DataFrame(resultados_lr)
display(tabla_lr[["prueba","accuracy","precision","recall","f1"]].round(4))

graficar_historias(
    historias_lr, "val_loss",
    "Impacto del learning rate - Pérdida de validación"
)

mejor_lr = float(tabla_lr.loc[tabla_lr["f1"].idxmax(), "lr"])
mejor_f1_lr = tabla_lr["f1"].max()

display(Markdown(
    f"**Análisis:** el learning rate **{mejor_lr}** obtuvo el mejor F1-Score de validación "
    f"(**{mejor_f1_lr:.4f}**). Un learning rate demasiado alto puede volver inestable el "
    "entrenamiento y uno demasiado bajo puede hacer más lenta la convergencia."
))


## 9. Comparación de funciones de activación

Se comparan `ReLU`, `tanh` y `sigmoid` en las capas ocultas. La capa de salida se mantiene con `sigmoid`, porque el problema continúa siendo una clasificación binaria.

La comparación considera tanto métricas como curvas de pérdida de validación, para analizar no solo el resultado final sino también la convergencia.


In [ ]:
resultados_activacion = []
historias_activacion = {}

for act in ["relu", "tanh", "sigmoid"]:
    r, h = ejecutar_prueba(
        nombre=f"activacion_{act}",
        epochs=mejor_epochs,
        batch_size=mejor_batch,
        learning_rate=mejor_lr,
        activacion=act,
        perdida="binary_crossentropy",
        optimizador="adam"
    )
    resultados_activacion.append(r)
    historias_activacion[act] = h

tabla_activacion = pd.DataFrame(resultados_activacion)
display(tabla_activacion[["prueba","accuracy","precision","recall","f1"]].round(4))

graficar_historias(
    historias_activacion, "val_loss",
    "Funciones de activación - Pérdida de validación"
)

plt.figure(figsize=(8,4))
plt.bar(tabla_activacion["activacion"], tabla_activacion["f1"])
plt.ylabel("F1-Score")
plt.title("Comparación de activaciones por F1-Score")
plt.show()

mejor_activacion = tabla_activacion.loc[tabla_activacion["f1"].idxmax(), "activacion"]
fila_act = tabla_activacion.loc[tabla_activacion["f1"].idxmax()]

display(Markdown(
    f"**Análisis:** `{mejor_activacion}` obtuvo el mayor F1-Score (**{fila_act['f1']:.4f}**) "
    f"con Precision **{fila_act['precision']:.4f}** y Recall **{fila_act['recall']:.4f}**. "
    "Por eso se selecciona para las siguientes pruebas."
))


## 10. Comparación de funciones de pérdida

Se comparan `binary_crossentropy` y `mean_squared_error` manteniendo fija la arquitectura y la mejor activación obtenida.

`Binary Crossentropy` es la opción teóricamente más adecuada para clasificación binaria porque penaliza directamente la diferencia entre la probabilidad predicha y la clase real. MSE se incluye como comparación experimental.


In [ ]:
resultados_perdida = []
historias_perdida = {}

for loss in ["binary_crossentropy", "mean_squared_error"]:
    r, h = ejecutar_prueba(
        nombre=f"loss_{loss}",
        epochs=mejor_epochs,
        batch_size=mejor_batch,
        learning_rate=mejor_lr,
        activacion=mejor_activacion,
        perdida=loss,
        optimizador="adam"
    )
    resultados_perdida.append(r)
    historias_perdida[loss] = h

tabla_perdida = pd.DataFrame(resultados_perdida)
display(tabla_perdida[["prueba","accuracy","precision","recall","f1"]].round(4))

graficar_historias(
    historias_perdida, "val_loss",
    "Funciones de pérdida - Pérdida de validación"
)

plt.figure(figsize=(8,4))
plt.bar(tabla_perdida["perdida"], tabla_perdida["f1"])
plt.ylabel("F1-Score")
plt.title("Comparación de funciones de pérdida")
plt.xticks(rotation=15)
plt.show()

mejor_perdida = tabla_perdida.loc[tabla_perdida["f1"].idxmax(), "perdida"]
fila_loss = tabla_perdida.loc[tabla_perdida["f1"].idxmax()]

display(Markdown(
    f"**Análisis:** `{mejor_perdida}` obtuvo el mayor F1-Score de validación "
    f"(**{fila_loss['f1']:.4f}**). La selección final se fundamenta en la evidencia "
    "experimental y en la adecuación de la función al problema binario."
))


## 11. Optimización y regularización

### 11.1 Comparación de optimizadores

Se comparan Adam, RMSprop y SGD usando la misma arquitectura y los hiperparámetros seleccionados. Esto permite observar el efecto del algoritmo utilizado para actualizar los pesos de la red.


In [ ]:
resultados_opt = []
historias_opt = {}

for opt in ["adam", "rmsprop", "sgd"]:
    r, h = ejecutar_prueba(
        nombre=f"opt_{opt}",
        epochs=mejor_epochs,
        batch_size=mejor_batch,
        learning_rate=mejor_lr,
        activacion=mejor_activacion,
        perdida=mejor_perdida,
        optimizador=opt
    )
    resultados_opt.append(r)
    historias_opt[opt] = h

tabla_opt = pd.DataFrame(resultados_opt)
display(tabla_opt[["prueba","accuracy","precision","recall","f1"]].round(4))

graficar_historias(
    historias_opt, "val_loss",
    "Comparación de optimizadores - Pérdida de validación"
)

mejor_opt = tabla_opt.loc[tabla_opt["f1"].idxmax(), "optimizador"]
fila_opt = tabla_opt.loc[tabla_opt["f1"].idxmax()]

display(Markdown(
    f"**Análisis:** `{mejor_opt}` obtuvo el mayor F1-Score (**{fila_opt['f1']:.4f}**) "
    "entre los optimizadores probados, por lo que se utiliza en la siguiente etapa."
))


### 11.2 Comparación con y sin regularización

Se evalúa una versión sin regularización y otra que incorpora:

- **L2** en las capas Dense.
- **Batch Normalization** después de la primera capa.
- **Dropout de 30% y 20%**.

Estas técnicas buscan reducir el sobreajuste y mejorar la estabilidad y generalización. La comparación se realiza con métricas y curvas de validación.


In [ ]:
resultados_reg = []
historias_reg = {}

for reg in [False, True]:
    nombre = "con_regularizacion" if reg else "sin_regularizacion"
    r, h = ejecutar_prueba(
        nombre=nombre,
        epochs=mejor_epochs,
        batch_size=mejor_batch,
        learning_rate=mejor_lr,
        activacion=mejor_activacion,
        perdida=mejor_perdida,
        optimizador=mejor_opt,
        regularizacion=reg
    )
    resultados_reg.append(r)
    historias_reg[nombre] = h

tabla_reg = pd.DataFrame(resultados_reg)
display(tabla_reg[["prueba","accuracy","precision","recall","f1"]].round(4))

graficar_historias(
    historias_reg, "val_loss",
    "Regularización - Pérdida de validación"
)
graficar_historias(
    historias_reg, "val_accuracy",
    "Regularización - Accuracy de validación"
)

mejor_reg = bool(tabla_reg.loc[tabla_reg["f1"].idxmax(), "regularizacion"])

f1_sin = float(tabla_reg.loc[tabla_reg["regularizacion"] == False, "f1"].iloc[0])
f1_con = float(tabla_reg.loc[tabla_reg["regularizacion"] == True, "f1"].iloc[0])

if mejor_reg:
    texto_reg = (
        f"La regularización mejoró el F1 de {f1_sin:.4f} a {f1_con:.4f}, "
        "por lo que se conserva en el modelo final."
    )
else:
    texto_reg = (
        f"La regularización fue evaluada, pero en esta ejecución el F1 pasó de "
        f"{f1_sin:.4f} a {f1_con:.4f}. Por eso el modelo final utiliza la alternativa "
        "sin regularización, ya que presentó mejor desempeño de validación."
    )

display(Markdown("**Análisis:** " + texto_reg))


## 12. Resumen de todas las pruebas

La tabla siguiente permite comparar las distintas configuraciones. La elección de cada ajuste se realizó usando el conjunto de validación y manteniendo el conjunto de prueba sin utilizar hasta el final.


In [ ]:
tabla_total = pd.concat([
    tabla_epochs,
    tabla_batch,
    tabla_lr,
    tabla_activacion,
    tabla_perdida,
    tabla_opt,
    tabla_reg
], ignore_index=True)

columnas_resumen = [
    "prueba","epochs","batch","lr","activacion","perdida",
    "optimizador","regularizacion","accuracy","precision","recall","f1"
]

display(tabla_total[columnas_resumen].round(4))

plt.figure(figsize=(12,5))
plt.bar(tabla_total["prueba"], tabla_total["f1"])
plt.xticks(rotation=75)
plt.ylabel("F1-Score")
plt.title("F1-Score de las configuraciones probadas")
plt.tight_layout()
plt.show()


## 13. Modelo final

El modelo final se construye con la combinación de hiperparámetros seleccionada durante los experimentos de validación.

Para reducir el riesgo de sobreajuste se utiliza `EarlyStopping` con `restore_best_weights=True`. Si la pérdida de validación deja de mejorar durante varias épocas, el entrenamiento se detiene y se recuperan los mejores pesos observados.


In [ ]:
print("Configuración final")
print("-------------------")
print("Épocas máximas:", mejor_epochs)
print("Batch:", mejor_batch)
print("Learning rate:", mejor_lr)
print("Activación:", mejor_activacion)
print("Pérdida:", mejor_perdida)
print("Optimizador:", mejor_opt)
print("Regularización:", mejor_reg)

tf.keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

modelo_final = crear_modelo(
    activacion=mejor_activacion,
    learning_rate=mejor_lr,
    perdida=mejor_perdida,
    optimizador=mejor_opt,
    regularizacion=mejor_reg
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

hist_final = modelo_final.fit(
    X_train_proc, y_train,
    validation_data=(X_val_proc, y_val),
    epochs=mejor_epochs,
    batch_size=mejor_batch,
    class_weight=pesos_clase,
    callbacks=[early_stopping],
    verbose=0
)

print("Épocas realmente ejecutadas:", len(hist_final.history["loss"]))


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(hist_final.history["accuracy"], label="Entrenamiento")
plt.plot(hist_final.history["val_accuracy"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Modelo final - Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(8,4))
plt.plot(hist_final.history["loss"], label="Entrenamiento")
plt.plot(hist_final.history["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Modelo final - Pérdida")
plt.legend()
plt.show()


## 14. Evaluación final en el conjunto de prueba

Las métricas principales son:

- **Accuracy:** proporción de predicciones correctas sobre el total.
- **Precision:** de los clientes predichos como `yes`, qué proporción realmente era `yes`.
- **Recall:** de todos los clientes que realmente eran `yes`, qué proporción fue detectada.
- **F1-Score:** media armónica entre Precision y Recall.

Debido al desbalance, Accuracy no se interpreta de forma aislada.


In [ ]:
prob_test = modelo_final.predict(X_test_proc, verbose=0).ravel()
pred_test = (prob_test >= 0.5).astype(int)

accuracy_final = accuracy_score(y_test, pred_test)
precision_final = precision_score(y_test, pred_test, zero_division=0)
recall_final = recall_score(y_test, pred_test, zero_division=0)
f1_final = f1_score(y_test, pred_test, zero_division=0)

metricas_finales = pd.DataFrame({
    "Métrica": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Valor": [accuracy_final, precision_final, recall_final, f1_final]
})

display(metricas_finales.round(4))
print("\nReporte de clasificación:")
print(classification_report(y_test, pred_test, target_names=["no","yes"], digits=4))


### 14.1 Matriz de confusión y cálculo de métricas

La matriz de confusión permite identificar:

- **VN:** verdaderos negativos.
- **FP:** falsos positivos.
- **FN:** falsos negativos.
- **VP:** verdaderos positivos.

A partir de esos cuatro valores se muestran también las fórmulas y un ejemplo numérico usando los resultados reales del modelo.


In [ ]:
cm = confusion_matrix(y_test, pred_test)
tn, fp, fn, tp = cm.ravel()

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["no","yes"]
).plot()
plt.title("Matriz de confusión - Modelo final")
plt.show()

total = tn + fp + fn + tp

acc_manual = (tp + tn) / total
prec_manual = tp / (tp + fp) if (tp + fp) else 0
rec_manual = tp / (tp + fn) if (tp + fn) else 0
f1_manual = (
    2 * prec_manual * rec_manual / (prec_manual + rec_manual)
    if (prec_manual + rec_manual) else 0
)

display(pd.DataFrame({
    "Valor": [tn, fp, fn, tp]
}, index=["VN","FP","FN","VP"]))

display(Markdown(
    f'''
**Ejemplo con la matriz obtenida:**

- Accuracy = (VP + VN) / Total = ({tp} + {tn}) / {total} = **{acc_manual:.4f}**
- Precision = VP / (VP + FP) = {tp} / ({tp} + {fp}) = **{prec_manual:.4f}**
- Recall = VP / (VP + FN) = {tp} / ({tp} + {fn}) = **{rec_manual:.4f}**
- F1 = 2 × (Precision × Recall) / (Precision + Recall) = **{f1_manual:.4f}**
'''
))


### 14.2 Comparación contra un baseline

Como referencia se compara el MLP contra un modelo trivial que predice siempre `no`. Esto permite demostrar por qué un Accuracy alto no es suficiente en un conjunto desbalanceado.


In [ ]:
pred_baseline = np.zeros(len(y_test), dtype=int)

baseline = {
    "accuracy": accuracy_score(y_test, pred_baseline),
    "precision": precision_score(y_test, pred_baseline, zero_division=0),
    "recall": recall_score(y_test, pred_baseline, zero_division=0),
    "f1": f1_score(y_test, pred_baseline, zero_division=0)
}

comparacion_final = pd.DataFrame([
    {"modelo":"Baseline siempre no", **baseline},
    {
        "modelo":"MLP final",
        "accuracy":accuracy_final,
        "precision":precision_final,
        "recall":recall_final,
        "f1":f1_final
    }
])

display(comparacion_final.round(4))

plt.figure(figsize=(8,4))
x = np.arange(4)
ancho = 0.35
metricas = ["accuracy","precision","recall","f1"]

plt.bar(x - ancho/2, comparacion_final.loc[0, metricas], ancho, label="Baseline")
plt.bar(x + ancho/2, comparacion_final.loc[1, metricas], ancho, label="MLP final")
plt.xticks(x, ["Accuracy","Precision","Recall","F1"])
plt.ylabel("Valor")
plt.title("Baseline vs MLP final")
plt.legend()
plt.show()


## 15. Interpretación de resultados y mejoras

La interpretación debe considerar el objetivo del negocio: identificar clientes con mayor probabilidad de contratar un depósito a plazo.

En este contexto:

- Un **falso negativo** corresponde a un cliente que sí habría contratado, pero el modelo no lo detectó.
- Un **falso positivo** corresponde a un cliente que el modelo marcó como potencial interesado, pero finalmente no contrató.
- Un Recall más alto ayuda a detectar más clientes positivos.
- Una Precision más alta reduce contactos dirigidos a clientes que finalmente no contratarían.

Por eso se revisa el equilibrio entre ambas métricas mediante F1-Score.


In [ ]:
# Diferencia entre Accuracy de entrenamiento y validación al final
acc_train_ult = hist_final.history["accuracy"][-1]
acc_val_ult = hist_final.history["val_accuracy"][-1]
gap_acc = acc_train_ult - acc_val_ult

display(Markdown(
    f'''
### Lectura de los resultados reales

La configuración final fue:

- **Épocas máximas:** {mejor_epochs}
- **Batch size:** {mejor_batch}
- **Learning rate:** {mejor_lr}
- **Activación:** `{mejor_activacion}`
- **Pérdida:** `{mejor_perdida}`
- **Optimizador:** `{mejor_opt}`
- **Regularización:** {mejor_reg}

En prueba obtuvo:

- **Accuracy:** {accuracy_final:.4f}
- **Precision:** {precision_final:.4f}
- **Recall:** {recall_final:.4f}
- **F1-Score:** {f1_final:.4f}

La diferencia final entre Accuracy de entrenamiento y validación fue **{gap_acc:.4f}**. 
Este valor, junto con las curvas de pérdida y Accuracy, se utiliza para revisar si existe una señal importante de sobreajuste.

Comparado con el baseline que siempre predice `no`, el MLP permite detectar casos positivos y por eso Recall y F1 son fundamentales para demostrar que la red realmente aprendió información útil.
'''
))


## 16. Conclusiones

El trabajo siguió un proceso completo de Deep Learning para clasificación tabular: carga de datos, preprocesamiento sin fuga de información, diseño de una MLP, entrenamiento, experimentos controlados, comparación de funciones, optimización, regularización y evaluación final.

Las decisiones más importantes fueron excluir `duration` por fuga de información, tratar correctamente el significado especial de `pdays = -1`, conservar `"unknown"` como categoría, utilizar estratificación y pesos de clase, y seleccionar los hiperparámetros únicamente con validación.

La configuración final no se eligió solo por Accuracy. Debido al desbalance, se utilizaron Precision, Recall y especialmente F1-Score para comparar alternativas.

El conjunto de prueba fue reservado hasta el final, por lo que sus métricas representan una estimación más confiable de la capacidad de generalización del modelo.


In [ ]:
display(Markdown(
    f'''
### Conclusión basada en esta ejecución

El modelo final obtuvo un **F1-Score de {f1_final:.4f}**, un **Recall de {recall_final:.4f}**
y una **Precision de {precision_final:.4f}** para la clase positiva.

Esto significa que el modelo no se limita a aprovechar la clase mayoritaria, sino que intenta identificar
a los clientes que realmente podrían contratar el depósito. La comparación con el baseline y la matriz de
confusión permiten evidenciar este punto.

Como mejora futura se podrían explorar ajustes del umbral de clasificación, búsqueda más amplia de
hiperparámetros o validación cruzada, siempre manteniendo el conjunto de prueba separado.
'''
))


## 17. Reproducibilidad y entrega

Para entregar el trabajo:

1. Ejecutar **Entorno de ejecución → Ejecutar todo** en Google Colab.
2. Verificar que no quede ninguna celda con error.
3. Guardar el notebook **con todos los resultados, tablas y gráficos visibles**.
4. Descargar el archivo `.ipynb`.
5. Subir el notebook al repositorio GitHub del equipo y compartir el enlace solicitado.
6. Preparar la defensa usando los resultados reales obtenidos en este notebook.

### Referencias técnicas

- UCI Machine Learning Repository — Bank Marketing.
- TensorFlow/Keras — documentación oficial de modelos Sequential, capas Dense, Dropout, BatchNormalization y optimizadores.
- Scikit-learn — documentación oficial de preprocesamiento y métricas de clasificación.
